In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

import time

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

ESTRATEGIA_DE_REBALANCEO = "NONE"
MODELO = "logreg"

NOMBRE_EXPERIMENTO = f"BCCC17__split__v1__{ESTRATEGIA_DE_REBALANCEO}_pca4_{MODELO}__v1"
CARPETA_DATASET = "BCCC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG LOGISTIC REGRESSION =====
LOGREG_C = 1.0
LOGREG_MAX_ITER = 1000
LOGREG_SOLVER = "lbfgs"
LOGREG_CLASS_WEIGHT = None
LOGREG_N_JOBS = -1

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 1

# ===== CONFIG REBALANCEO DENTRO DEL CV =====
TARGET_N = 10000
NEARMISS_VERSION = 1
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS = 3

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(878920, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,80,0,0.000000,1,0,0,0,0.000000,0.000000e+00,0.000000,...,0,0,0,1.499262e+09,1.499262e+09,0.000000,0.000000,0.000000,0.000000,1
1,53,1,0.062127,4,64,55,32,43.500000,1.322500e+02,0.000000,...,0,0,0,2.070904e-02,9.500000e-07,0.000000,0.000003,0.000000,0.000000,0
2,80,0,9.282073,15,20,11595,0,774.333333,8.363370e+06,36.000000,...,1,0,0,6.630052e-01,1.031341e+00,2.841435,0.045323,10.000000,20.000000,3
3,443,0,0.000000,1,0,0,0,0.000000,0.000000e+00,0.000000,...,0,0,0,1.499098e+09,1.499098e+09,0.000000,0.000000,0.000000,0.000000,0
4,22,0,12.005657,54,2008,976,0,88.018519,3.527883e+04,18226.380165,...,1,0,0,2.265218e-01,5.716980e-01,0.900982,0.387251,3.666667,334.666667,7


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,360000
1,276914
2,129058
3,76583
4,7625
5,6691
6,5485
7,4759
8,4406


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (878920, 63)
Shape y_train: (878920,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("logreg", LogisticRegression(
        C=LOGREG_C,
        max_iter=LOGREG_MAX_ITER,
        solver=LOGREG_SOLVER,
        class_weight=LOGREG_CLASS_WEIGHT,
        n_jobs=LOGREG_N_JOBS,
        random_state=RANDOM_STATE
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",1
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
labels_globales = np.array(sorted(y_train.unique()))

resultados_folds = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):

    print("=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 80)

    # =========================
    # Split del fold
    # =========================
    df_train_fold = df_train.iloc[train_idx].copy()
    df_val_fold = df_train.iloc[val_idx].copy()

    print("Shape train fold original:", df_train_fold.shape)
    print("Shape val fold original  :", df_val_fold.shape)
    print()

    # =========================
    # Rebalanceo SOLO sobre train fold
    # =========================
    df_train_fold_balanceado = rebalancear_train_fold(
        df_fold_train=df_train_fold,
        label_col=LABEL_COL,
        target_n=TARGET_N,
        random_state=RANDOM_STATE + fold,
        nearmiss_version=NEARMISS_VERSION,
        smote_k_neighbors=SMOTE_K_NEIGHBORS,
        estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
        enn_n_neighbors=ENN_N_NEIGHBORS
    )

    X_train_fold_bal = df_train_fold_balanceado.drop(columns=[LABEL_COL])
    y_train_fold_bal = df_train_fold_balanceado[LABEL_COL]

    X_val_fold = df_val_fold.drop(columns=[LABEL_COL])
    y_val_fold = df_val_fold[LABEL_COL]

    # =========================
    # Modelo nuevo para cada fold
    # =========================
    pipeline_fold = Pipeline([
        ("scaler", RobustScaler()),
        ("pca", PCA(n_components=N_COMPONENTS_PCA)),
        ("logreg", LogisticRegression(
            C=LOGREG_C,
            max_iter=LOGREG_MAX_ITER,
            solver=LOGREG_SOLVER,
            class_weight=LOGREG_CLASS_WEIGHT,
            n_jobs=LOGREG_N_JOBS,
            random_state=RANDOM_STATE + fold
        ))
    ])

    # =========================
    # Entrenamiento
    # =========================
    t0 = time.time()
    pipeline_fold.fit(X_train_fold_bal, y_train_fold_bal)
    fit_time = time.time() - t0

    # =========================
    # Validación
    # =========================
    t0 = time.time()
    y_pred_val = pipeline_fold.predict(X_val_fold)
    score_time = time.time() - t0

    roc_auc_val = calcular_roc_auc_multiclase_seguro(
        modelo=pipeline_fold,
        X_val=X_val_fold,
        y_val=y_val_fold,
        labels_globales=labels_globales
    )

    metricas_fold = {
        "fold": fold,

        "train_original_rows": int(df_train_fold.shape[0]),
        "train_balanceado_rows": int(df_train_fold_balanceado.shape[0]),
        "val_rows": int(df_val_fold.shape[0]),

        "accuracy": accuracy_score(y_val_fold, y_pred_val),

        "precision_weighted": precision_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_val_fold, y_pred_val, average="weighted", zero_division=0
        ),

        "precision_macro": precision_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_val_fold, y_pred_val, average="macro", zero_division=0
        ),

        "mcc": matthews_corrcoef(y_val_fold, y_pred_val),
        "roc_auc": roc_auc_val,

        "fit_time": fit_time,
        "score_time": score_time
    }

    resultados_folds.append(metricas_fold)

    print("Métricas fold:")
    print(metricas_fold)
    print()

FOLD 1/5


Shape train fold original: (703136, 64)
Shape val fold original  : (175784, 64)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     221531
2     103247
3      61266
4       6100
5       5353
6       4388
7       3807
8       3525
9       3278
10      1748
11       869
12        16
13         8
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     221531
2     103247
3      61266
4       6100
5       5353
6       4388
7       3807
8       3525
9       3278
10      1748
11       869
12        16
13         8
Name: count, dtype: int64
Shape final: (703136, 64)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 1, 'train_original_rows': 703136, 'train_balanceado_rows': 703136, 'val_rows': 175784, 'accuracy': 0.25750921585582304, 'precision_weighted': 0.18789534453135973, 'recall_weighted': 0.25750921585582304, 'f1_weighted': 0.178357331294333, 'precision_macro': 0.04969088293043215, 'recall_macro': 0.09601028808674561, 'f1_macro': 0.05237149627538546, 'mcc': 0.1493347339077301, 'roc_auc': nan, 'fit_time': 3.6606929302215576, 'score_time': 0.05967545509338379}

FOLD 2/5


Shape train fold original: (703136, 64)
Shape val fold original  : (175784, 64)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     221531
2     103247
3      61266
4       6100
5       5352
6       4388
7       3808
8       3525
9       3278
10      1749
11       869
12        15
13         8
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     221531
2     103247
3      61266
4       6100
5       5352
6       4388
7       3808
8       3525
9       3278
10      1749
11       869
12        15
13         8
Name: count, dtype: int64
Shape final: (703136, 64)



...

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 4, 'train_original_rows': 703136, 'train_balanceado_rows': 703136, 'val_rows': 175784, 'accuracy': 0.25753197105538617, 'precision_weighted': 0.1877181067757897, 'recall_weighted': 0.25753197105538617, 'f1_weighted': 0.17833400751834644, 'precision_macro': 0.04965268237162541, 'recall_macro': 0.0960082840674324, 'f1_macro': 0.05236865333355245, 'mcc': 0.14917623024752918, 'roc_auc': nan, 'fit_time': 3.6009058952331543, 'score_time': 0.05264878273010254}

FOLD 5/5


Shape train fold original: (703136, 64)
Shape val fold original  : (175784, 64)

Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     288000
1     221532
2     103246
3      61266
4       6100
5       5353
6       4388
7       3807
8       3524
9       3279
10      1749
11       869
12        15
13         8
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     288000
1     221532
2     103246
3      61266
4       6100
5       5353
6       4388
7       3807
8       3524
9       3279
10      1749
11       869
12        15
13         8
Name: count, dtype: int64
Shape final: (703136, 64)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


No se pudo calcular ROC AUC en este fold: name 'roc_auc_score' is not defined
Métricas fold:
{'fold': 5, 'train_original_rows': 703136, 'train_balanceado_rows': 703136, 'val_rows': 175784, 'accuracy': 0.2606153005961862, 'precision_weighted': 0.19010460218992106, 'recall_weighted': 0.2606153005961862, 'f1_weighted': 0.181368437288497, 'precision_macro': 0.050221089170584335, 'recall_macro': 0.09671221883016298, 'f1_macro': 0.05309514132001679, 'mcc': 0.15377607670249474, 'roc_auc': nan, 'fit_time': 3.5126755237579346, 'score_time': 0.05890917778015137}



In [11]:
df_folds = pd.DataFrame(resultados_folds)

df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,703136,703136,175784,0.257509,0.187895,0.257509,0.178357,0.049691,0.096010,0.052371,0.149335,NaN,3.660693,0.059675
1,2,703136,703136,175784,0.258232,0.188823,0.258232,0.179200,0.049901,0.096162,0.052562,0.150669,NaN,3.675126,0.052721
2,3,703136,703136,175784,0.258909,0.189089,0.258909,0.179817,0.049969,0.096303,0.052712,0.151405,NaN,3.550870,0.061138
3,4,703136,703136,175784,0.257532,0.187718,0.257532,0.178334,0.049653,0.096008,0.052369,0.149176,NaN,3.600906,0.052649
4,5,703136,703136,175784,0.260615,0.190105,0.260615,0.181368,0.050221,0.096712,0.053095,0.153776,NaN,3.512676,0.058909


In [12]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "modelo": "LogisticRegression",
        "logreg_c": LOGREG_C,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_solver": LOGREG_SOLVER,
        "logreg_class_weight": LOGREG_CLASS_WEIGHT,
        "logreg_n_jobs": LOGREG_N_JOBS,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'BCCC17__split__v1__NONE_pca4_logreg__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv',
 'shape_train': {'rows': 878920, 'cols': 64},
 'parametros': {'modelo': 'LogisticRegression',
  'logreg_c': 1.0,
  'logreg_max_iter': 1000,
  'logreg_solver': 'lbfgs',
  'logreg_class_weight': None,
  'logreg_n_jobs': -1,
  'n_components_pca': 1,
  'estrategia_rebalanceo': 'NONE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_media': {'accuracy': 0.25855936831566007,
  'precision_weighted': 0.18872615055275913,
  'recall_weighted': 0.25855936831566007,
  'f1_weighted': 0.1794153525957894,
  'precision_macro': 0.04988696111569606,
  'recall_macro': 0.09623914363561512,
  'f1_macro': 0.052621746515176385,
  'mcc': 0.15087233970084718,
  'roc_auc': nan,
  'fit_time': 3.600054121017456,
  'score_time': 0.057018375396728514},
 'metricas_std'

In [13]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}\t"
    f"{summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}\t"
    f"{summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}\t"
    f"{summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.258559 ± 0.001286	0.188726 ± 0.000968	0.258559 ± 0.001286	0.179415 ± 0.001256	0.049887 ± 0.000230	0.096239 ± 0.000291	0.052622 ± 0.000301	0.150872 ± 0.001871	nan ± nan


In [14]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1/BCCC17__split__v1__NONE_pca4_logreg__v1__folds.csv


In [15]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1/BCCC17__split__v1__NONE_pca4_logreg__v1__summary_cv.json


In [16]:
df_folds

,fold,train_original_rows,train_balanceado_rows,val_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,703136,703136,175784,0.257509,0.187895,0.257509,0.178357,0.049691,0.096010,0.052371,0.149335,NaN,3.660693,0.059675
1,2,703136,703136,175784,0.258232,0.188823,0.258232,0.179200,0.049901,0.096162,0.052562,0.150669,NaN,3.675126,0.052721
2,3,703136,703136,175784,0.258909,0.189089,0.258909,0.179817,0.049969,0.096303,0.052712,0.151405,NaN,3.550870,0.061138
3,4,703136,703136,175784,0.257532,0.187718,0.257532,0.178334,0.049653,0.096008,0.052369,0.149176,NaN,3.600906,0.052649
4,5,703136,703136,175784,0.260615,0.190105,0.260615,0.181368,0.050221,0.096712,0.053095,0.153776,NaN,3.512676,0.058909


In [17]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(219731, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,714,0,0.000082,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,8.202000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
1,80,0,0.172038,12,396,7240,0,999.25,4.955239e+06,19201.959184,...,1,0,0,1.563983e-02,2.867301e-02,0.058456,4.013026e-02,0.0,0.0,1
2,3003,0,0.000043,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,4.292000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
3,3389,0,0.000098,2,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,1,9.799000e-05,1.499450e+09,0.000000,1.499450e+09,0.0,0.0,2
4,80,0,0.000000,1,0,0,0,0.00,0.000000e+00,0.000000,...,0,0,0,1.499344e+09,1.499344e+09,0.000000,0.000000e+00,0.0,0.0,10


In [18]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,90000
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [19]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (219731, 63)
Shape y_test: (219731,)


In [20]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [21]:
print("Rebalanceando todo el train original para entrenar el modelo final...")

df_train_balanceado_final = rebalancear_train_fold(
    df_fold_train=df_train,
    label_col=LABEL_COL,
    target_n=TARGET_N,
    random_state=RANDOM_STATE,
    nearmiss_version=NEARMISS_VERSION,
    smote_k_neighbors=SMOTE_K_NEIGHBORS,
    estrategia_rebalanceo=ESTRATEGIA_DE_REBALANCEO,
    enn_n_neighbors=ENN_N_NEIGHBORS
)

X_train_balanceado_final = df_train_balanceado_final.drop(columns=[LABEL_COL])
y_train_balanceado_final = df_train_balanceado_final[LABEL_COL]

pipeline.fit(X_train_balanceado_final, y_train_balanceado_final)

print("Modelo final entrenado con todo el train rebalanceado.")
print("Train original   :", df_train.shape)
print("Train balanceado :", df_train_balanceado_final.shape)

Rebalanceando todo el train original para entrenar el modelo final...


Estrategia de rebalanceo: NONE
Distribución antes del rebalanceo:
LABEL
0     360000
1     276914
2     129058
3      76583
4       7625
5       6691
6       5485
7       4759
8       4406
9       4098
10      2186
11      1086
12        19
13        10
Name: count, dtype: int64

No se aplica ningún rebalanceo.
Distribución final:
LABEL
0     360000
1     276914
2     129058
3      76583
4       7625
5       6691
6       5485
7       4759
8       4406
9       4098
10      2186
11      1086
12        19
13        10
Name: count, dtype: int64
Shape final: (878920, 64)



/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Modelo final entrenado con todo el train rebalanceado.
Train original   : (878920, 64)
Train balanceado : (878920, 64)


In [22]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 219731


In [23]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.25866172729382747,
 'precision_weighted': 0.18879503313009063,
 'recall_weighted': 0.25866172729382747,
 'f1_weighted': 0.17948096115546755,
 'precision_macro': 0.04990567893586843,
 'recall_macro': 0.09627906721618688,
 'f1_macro': 0.05264191772732307,
 'roc_auc': 0.5648499627758736,
 'mcc': 0.15108104203774278}

In [24]:
print("Accuracy\tPrecision weighted\tRecall weighted\tF1 weighted\tPrecision macro\tRecall macro\tF1 macro\tMCC\tROC AUC")

print(
    f"{metricas_test['accuracy']:.6f}\t"
    f"{metricas_test['precision_weighted']:.6f}\t"
    f"{metricas_test['recall_weighted']:.6f}\t"
    f"{metricas_test['f1_weighted']:.6f}\t"
    f"{metricas_test['precision_macro']:.6f}\t"
    f"{metricas_test['recall_macro']:.6f}\t"
    f"{metricas_test['f1_macro']:.6f}\t"
    f"{metricas_test['mcc']:.6f}\t"
    f"{metricas_test['roc_auc']:.6f}"
)

Accuracy	Precision weighted	Recall weighted	F1 weighted	Precision macro	Recall macro	F1 macro	MCC	ROC AUC
0.258662	0.188795	0.258662	0.179481	0.049906	0.096279	0.052642	0.151081	0.564850


In [25]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0,20934,69066,0,0,0,0,0,0,0,0,0,0,0
1,0,24995,44234,0,0,0,0,0,0,0,0,0,0,0
2,0,424,31841,0,0,0,0,0,0,0,0,0,0,0
3,0,11,19135,0,0,0,0,0,0,0,0,0,0,0
4,0,634,1272,0,0,0,0,0,0,0,0,0,0,0
5,0,94,1579,0,0,0,0,0,0,0,0,0,0,0
6,0,357,1014,0,0,0,0,0,0,0,0,0,0,0
7,0,570,620,0,0,0,0,0,0,0,0,0,0,0
8,0,147,955,0,0,0,0,0,0,0,0,0,0,0
9,0,191,833,0,0,0,0,0,0,0,0,0,0,0


In [26]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     90000
           1       0.51      0.36      0.42     69229
           2       0.19      0.99      0.31     32265
           3       0.00      0.00      0.00     19146
           4       0.00      0.00      0.00      1906
           5       0.00      0.00      0.00      1673
           6       0.00      0.00      0.00      1371
           7       0.00      0.00      0.00      1190
           8       0.00      0.00      0.00      1102
           9       0.00      0.00      0.00      1024
          10       0.00      0.00      0.00       547
          11       0.00      0.00      0.00       271
          12       0.00      0.00      0.00         5
          13       0.00      0.00      0.00         2

    accuracy                           0.26    219731
   macro avg       0.05      0.10      0.05    219731
weighted avg       0.19      0.

In [27]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "modelo": "LogisticRegression",
        "logreg_c": LOGREG_C,
        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_solver": LOGREG_SOLVER,
        "logreg_class_weight": LOGREG_CLASS_WEIGHT,
        "logreg_n_jobs": LOGREG_N_JOBS,
        "n_components_pca": N_COMPONENTS_PCA,
        "estrategia_rebalanceo": ESTRATEGIA_DE_REBALANCEO,
        "target_n": TARGET_N,
        "nearmiss_version": NEARMISS_VERSION,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "enn_n_neighbors": ENN_N_NEIGHBORS
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'BCCC17__split__v1__NONE_pca4_logreg__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv',
 'shape_test': {'rows': 219731, 'cols': 64},
 'parametros': {'modelo': 'LogisticRegression',
  'logreg_c': 1.0,
  'logreg_max_iter': 1000,
  'logreg_solver': 'lbfgs',
  'logreg_class_weight': None,
  'logreg_n_jobs': -1,
  'n_components_pca': 1,
  'estrategia_rebalanceo': 'NONE',
  'target_n': 10000,
  'nearmiss_version': 1,
  'smote_k_neighbors': 5,
  'enn_n_neighbors': 3},
 'metricas_test': {'accuracy': 0.25866172729382747,
  'precision_weighted': 0.18879503313009063,
  'recall_weighted': 0.25866172729382747,
  'f1_weighted': 0.17948096115546755,
  'precision_macro': 0.04990567893586843,
  'recall_macro': 0.09627906721618688,
  'f1_macro': 0.05264191772732307,
  'mcc': 0.15108104203774278}}

In [28]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1/BCCC17__split__v1__NONE_pca4_logreg__v1__metricas_test.csv


In [29]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1/BCCC17__split__v1__NONE_pca4_logreg__v1__confusion_matrix_test.csv


In [30]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES_MEJORADO2/04_experimentos/logs/resultados/BCCC17__split__v1__NONE_pca4_logreg__v1/BCCC17__split__v1__NONE_pca4_logreg__v1__summary_test.json


In [31]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.25855936831566007, 'precision_weighted': 0.18872615055275913, 'recall_weighted': 0.25855936831566007, 'f1_weighted': 0.1794153525957894, 'precision_macro': 0.04988696111569606, 'recall_macro': 0.09623914363561512, 'f1_macro': 0.052621746515176385, 'mcc': 0.15087233970084718, 'roc_auc': nan, 'fit_time': 3.600054121017456, 'score_time': 0.057018375396728514}

TEST:
{'accuracy': 0.25866172729382747, 'precision_weighted': 0.18879503313009063, 'recall_weighted': 0.25866172729382747, 'f1_weighted': 0.17948096115546755, 'precision_macro': 0.04990567893586843, 'recall_macro': 0.09627906721618688, 'f1_macro': 0.05264191772732307, 'mcc': 0.15108104203774278}
